# 🌍 Atmosphäre-Analyse
## Layer 1 – Planetarer Grundkörper

| Layer | Name | Status |
|-------|------|--------|
| 0 | Externe kosmische Einflüsse | ✅ `layer0_state.json` |
| **1** | **Planetarer Grundkörper** | **← dieser Layer** |
| 2 | Erdoberfläche / Ozeane / Land | ⬜ |
| 3 | Atmosphäre / Wetter / Gewitter | ⬜ |
| 4 | Ionosphäre | ⬜ |
| 5 | Global Electric Circuit | ⬜ |
| 6 | Resonanz- und Musterfeld | ⬜ |
| 7 | Interpretation / Systemzustand | ⬜ |

> **Kernidee:** Ohne den planetaren Körper gäbe es keinen Resonanzraum, keine globale elektrische Struktur und keine Kopplungsschichten.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math, requests
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

# --- Projektpfade (CWD-unabhaengig, ohne pip install) ---
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / '.project-root').exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / 'src'))
from atmosphere.paths import layer_state

print(f'✅ Pakete geladen')
print(f'📅 Analysedatum: {datetime.date.today()}')

# Layer-0-Kontext laden
try:
    with open(layer_state(0), encoding='utf-8') as f:
        layer0 = json.load(f)
    print(f'\n📥 Layer 0 geladen:')
    print(f'   Score:   {layer0["score"]}  |  Level: {layer0["level"].upper()}')
    print(f'   Driver:  {layer0["dominant_driver"]}')
    print(f'   Summary: {layer0["state_summary"][:120]}...')
except FileNotFoundError:
    print('⚠️  layer0_state.json nicht gefunden – Layer-0-Kontext fehlt')
    layer0 = None

---
## 1. Systemstruktur: Die 6 Elemente von Layer 1

In [ ]:
elements = [
    {'name': 'Erdmasse<br>& Schwere',         'x': 0.50, 'y': 0.90, 'color': '#888780', 'sz': 56},
    {'name': 'Erdrotation<br>(LOD)',           'x': 0.18, 'y': 0.73, 'color': '#5F5E5A', 'sz': 54},
    {'name': 'Erdmagnetfeld<br>(IGRF)',         'x': 0.82, 'y': 0.73, 'color': '#534AB7', 'sz': 56},
    {'name': 'Lithosphäre<br>& Seismik',       'x': 0.14, 'y': 0.50, 'color': '#B4B2A9', 'sz': 54},
    {'name': 'Leitfähigkeit<br>Erdoberfläche', 'x': 0.86, 'y': 0.50, 'color': '#639922', 'sz': 54},
    {'name': 'Ozeane als<br>Leitfähigkörper',  'x': 0.50, 'y': 0.53, 'color': '#378ADD', 'sz': 56},
]
core = {'x': 0.50, 'y': 0.17}

fig = go.Figure()
for el in elements:
    dx = core['x'] - el['x']
    dy = core['y'] - el['y']
    dist = math.sqrt(dx**2 + dy**2)
    t = 0.055 / dist
    ex, ey = core['x'] - dx * t, core['y'] - dy * t
    fig.add_trace(go.Scatter(
        x=[el['x'], ex], y=[el['y'], ey], mode='lines',
        line=dict(color=el['color'], width=1.8), opacity=0.45,
        showlegend=False, hoverinfo='skip'
    ))

# Kern / Erdsystem
fig.add_trace(go.Scatter(
    x=[core['x']], y=[core['y']], mode='markers+text',
    marker=dict(size=84, color='#3d3d3a', opacity=0.92,
                line=dict(color='#B4B2A9', width=2.5)),
    text=['🌍 Planetarer<br>Grundkörper'], textposition='middle center',
    textfont=dict(size=10, color='white'), showlegend=False, hoverinfo='skip'
))
for el in elements:
    fig.add_trace(go.Scatter(
        x=[el['x']], y=[el['y']], mode='markers+text',
        marker=dict(size=el['sz'], color=el['color'], opacity=0.90,
                    line=dict(color='white', width=2)),
        text=[el['name']], textposition='middle center',
        textfont=dict(size=9.5, color='white'),
        showlegend=False,
        hovertemplate=el['name'].replace('<br>',' ') + '<extra></extra>'
    ))

for txt, px, py, col in [
    ('⚫  Masse & Dynamik',          0.50, 0.99, '#5F5E5A'),
    ('🔵  Elektrisch / Magnetisch',  0.50, 0.61, '#185FA5'),
    ('🟤  Geophysikalische Struktur', 0.50, 0.40, '#3B6D11'),
]:
    fig.add_annotation(x=px, y=py, text=txt, showarrow=False,
                       xref='paper', yref='paper',
                       font=dict(size=11, color=col))

# Layer-0-Kontext einblenden
if layer0:
    ctx = f'Layer-0-Input: {layer0["level"].upper()} | Driver: {layer0["dominant_driver"]} | Confidence: {layer0["confidence"]:.0%}'
    fig.add_annotation(x=0.5, y=0.03, text=ctx, showarrow=False,
                       xref='paper', yref='paper',
                       font=dict(size=10, color='#888780'))

fig.update_layout(
    title=dict(text='Layer 1 – Planetarer Grundkörper: Systemstruktur', font=dict(size=16)),
    xaxis=dict(showgrid=False, zeroline=False, visible=False, range=[-0.05, 1.05]),
    yaxis=dict(showgrid=False, zeroline=False, visible=False, range=[0.02, 1.05]),
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    height=530, margin=dict(l=20, r=20, t=55, b=20)
)
fig.show()

---
## 2. Physikalische Konstanten & Referenzwerte

In [ ]:
# ============================================================
# LAYER 1 – PHYSIKALISCHE KONSTANTEN (zeitinvariant)
# Quelle: IERS Conventions 2010, WGS-84, IGRF-13
# ============================================================

CONSTANTS = {
    # Masse & Geometrie
    'Erdmasse_kg':            5.9722e24,
    'Erdradius_aequator_km':  6378.137,
    'Erdradius_pol_km':       6356.752,
    'Abplattung':             1 / 298.257,

    # Rotation
    'Siderischer_Tag_s':      86164.1,      # 23h 56min 4.1s
    'Winkelgeschw_rad_s':     7.2921150e-5, # Omega
    'Rotationsperiode_h':     23.9345,

    # Magnetfeld (IGRF-13 Dipolmoment)
    'Dipolmoment_Am2':        7.94e22,      # magnetisches Dipolmoment
    'Dipol_Neigung_Grad':     11.5,         # Winkel geographisch/magnetisch
    'Mag_Pol_Nord_lat':       86.5,         # ca. 2024
    'Mag_Pol_Nord_lon':       162.7,

    # Elektrische Eigenschaften
    'Leitf_Ozeane_Sm':        3.2,          # S/m (Meerwasser typ.)
    'Leitf_Kontinent_Sm':     1e-4,         # S/m (trockenes Gestein)
    'Leitf_Feuchter_Boden':   0.05,         # S/m
    'Schumann_Grundmode_Hz':  7.83,         # Hz (Referenz, ruhig)

    # Lithosphäre
    'Kruste_Dicke_km_ozean':  7,
    'Kruste_Dicke_km_kont':   35,
    'Mantel_Tiefe_km':        2890,
    'Kern_Tiefe_km':          5150,
}

print('LAYER 1 – PHYSIKALISCHE KONSTANTEN & REFERENZWERTE')
print('=' * 56)
groups = [
    ('Masse & Geometrie',   ['Erdmasse_kg','Erdradius_aequator_km','Erdradius_pol_km','Abplattung']),
    ('Rotation',            ['Siderischer_Tag_s','Winkelgeschw_rad_s','Rotationsperiode_h']),
    ('Magnetfeld (IGRF-13)',['Dipolmoment_Am2','Dipol_Neigung_Grad','Mag_Pol_Nord_lat','Mag_Pol_Nord_lon']),
    ('Leitfähigkeit',       ['Leitf_Ozeane_Sm','Leitf_Kontinent_Sm','Leitf_Feuchter_Boden','Schumann_Grundmode_Hz']),
    ('Struktur',            ['Kruste_Dicke_km_ozean','Kruste_Dicke_km_kont','Mantel_Tiefe_km','Kern_Tiefe_km']),
]
for grp, keys in groups:
    print(f'\n  {grp}')
    for k in keys:
        print(f'    {k:<35} {CONSTANTS[k]:>14g}')

---
## 3. Echtdaten abrufen

In [ ]:
# ============================================================
# ECHTDATEN – DREI QUELLEN
# 1) USGS  – Seismik (Erdbeben M≥4.5, letzte 7 Tage)
# 2) IERS  – Erdrotation (Length of Day, Polverlagerung)
# 3) NOAA  – Geomagnetische Feldvariationen (aus Layer-0-Daten)
# ============================================================

raw = {}

# --- 1) USGS Seismik ---
try:
    end   = datetime.date.today()
    start = end - datetime.timedelta(days=7)
    url = (f'https://earthquake.usgs.gov/fdsnws/event/1/query'
           f'?format=geojson&starttime={start}&endtime={end}'
           f'&minmagnitude=4.5&orderby=time')
    r = requests.get(url, timeout=20); r.raise_for_status()
    raw['seismic'] = r.json()
    n = len(raw['seismic']['features'])
    print(f'  ✅ USGS Seismik       {n:>5} Ereignisse (M≥4.5, 7 Tage)')
except Exception as e:
    print(f'  ❌ USGS Seismik       {e}')
    raw['seismic'] = None

# --- 2) IERS Erdrotation – mehrere Fallback-URLs ---
IERS_URLS = [
    # Primär: USNO (US Naval Observatory) – zuverlässigster Mirror
    'https://maia.usno.navy.mil/ser7/finals2000A.daily',
    # Fallback 1: Paris Observatory
    'https://hpiers.obspm.fr/iers/eop/eopc04/finals2000A.daily',
    # Fallback 2: IERS Data Center (Festbreiten)
    'https://datacenter.iers.org/data/9/finals2000A.daily',
    # Fallback 3: IERS Data Center CSV
    'https://datacenter.iers.org/data/csv/finals2000A.daily.csv',
]
raw['iers_raw'] = None
raw['iers_url'] = None
for url in IERS_URLS:
    try:
        r = requests.get(url, timeout=15); r.raise_for_status()
        if len(r.text) > 1000:
            raw['iers_raw'] = r.text
            raw['iers_url'] = url
            print(f'  ✅ IERS EOP           {len(r.text):>7} Zeichen  ({url.split("/")[-1]})')
            break
        else:
            print(f'  ⚠️  {url.split("/")[-1]} – Antwort zu kurz')
    except Exception as e:
        print(f'  ↩  {url.split("/")[-1]} – {str(e)[:70]}')
if raw['iers_raw'] is None:
    # Synthetischer LOD-Schätzwert: saisonale Variation ±1ms
    doy = datetime.date.today().timetuple().tm_yday
    lod_estimated = round(0.8 * math.sin(2 * math.pi * doy / 365.25), 3)
    raw['iers_synthetic_lod'] = lod_estimated
    print(f'  ⚠️  IERS nicht erreichbar → saisonaler Schätzwert: LOD ≈ {lod_estimated:+.3f} ms')

# --- 3) NOAA Geomagnetismus (Dst-Index als Proxy für Feldvariationen) ---
try:
    url = 'https://services.swpc.noaa.gov/json/planetary_k_index_1m.json'
    r = requests.get(url, timeout=15); r.raise_for_status()
    raw['geomag'] = r.json()
    print(f'  ✅ NOAA Geomag Kp     {len(raw["geomag"]):>5} Einträge')
except Exception as e:
    print(f'  ❌ NOAA Geomag        {e}')
    raw['geomag'] = None

print('\n📥 Datenabruf abgeschlossen')

In [ ]:
# ============================================================
# DATEN AUFBEREITEN
# ============================================================

# --- Seismik ---
df_seis = None
if raw['seismic']:
    features = raw['seismic']['features']
    rows = []
    for f in features:
        p = f['properties']
        g = f['geometry']['coordinates']
        rows.append({
            'time':  pd.to_datetime(p['time'], unit='ms'),
            'mag':   p['mag'],
            'place': p['place'],
            'depth': g[2],
            'lon':   g[0],
            'lat':   g[1],
        })
    df_seis = pd.DataFrame(rows).sort_values('time', ascending=False)
    print(f'Seismik:  {len(df_seis)} Ereignisse')
    print(f'  Stärkste: M{df_seis["mag"].max():.1f} – {df_seis.loc[df_seis["mag"].idxmax(), "place"]}')
    print(f'  Tiefste:  {df_seis["depth"].max():.0f} km')
    print(f'  M≥6.0:    {(df_seis["mag"] >= 6.0).sum()} Ereignisse')

# --- IERS Erdrotation ---
df_iers  = None
lod_col  = None
if raw['iers_raw']:
    try:
        text = raw['iers_raw']
        is_csv = ';' in text[:500] or ',' in text[:500]

        if is_csv:
            # CSV-Format (IERS Data Center)
            lines = [l for l in text.splitlines() if not l.startswith('#') and l.strip()]
            sep = ';' if ';' in lines[0] else ','
            header = [h.strip().lower().replace(' ','_') for h in lines[0].split(sep)]
            records = []
            for line in lines[1:]:
                parts = line.split(sep)
                if len(parts) >= len(header):
                    records.append(dict(zip(header, [p.strip() for p in parts])))
            df_iers = pd.DataFrame(records)
        else:
            # Festbreiten-Format (USNO finals2000A.daily)
            # Spalten: Jahr(2) Monat(2) Tag(2) MJD(8) x(9) xErr(9) y(9) yErr(9) UT1-UTC(10) ... LOD(7)
            rows = []
            for line in text.splitlines():
                if len(line) < 68: continue
                try:
                    rows.append({
                        'mjd':      float(line[7:15].strip())  if line[7:15].strip()  else None,
                        'pm_x':     float(line[18:27].strip()) if line[18:27].strip() else None,
                        'pm_y':     float(line[37:46].strip()) if line[37:46].strip() else None,
                        'ut1_utc':  float(line[58:68].strip()) if line[58:68].strip() else None,
                        'lod':      float(line[79:86].strip()) if len(line)>86 and line[79:86].strip() else None,
                    })
                except: continue
            df_iers = pd.DataFrame(rows)

        if df_iers is not None and not df_iers.empty:
            # LOD-Spalte finden
            lod_col = next((c for c in df_iers.columns if 'lod' in c.lower()), None)
            if lod_col:
                df_iers[lod_col] = pd.to_numeric(df_iers[lod_col], errors='coerce')
                df_iers = df_iers.dropna(subset=[lod_col])
                # LOD in ms umrechnen falls nötig (USNO liefert Sekunden)
                if df_iers[lod_col].abs().median() < 0.01:
                    df_iers[lod_col] = df_iers[lod_col] * 1000  # s → ms
                df_iers = df_iers.tail(90)
                print(f'IERS: {len(df_iers)} Zeilen, LOD aktuell {df_iers[lod_col].iloc[-1]:.4f} ms')
            else:
                print(f'  ⚠️  IERS Spalten: {df_iers.columns.tolist()} – keine LOD-Spalte gefunden')
    except Exception as e:
        print(f'  ⚠️  IERS Parsing Fehler: {e}')
        df_iers = None

# --- Geomagnetismus (Kp als dynamische Feldvariation) ---
df_geomag = None
if raw['geomag']:
    df_geomag = pd.DataFrame(raw['geomag'])
    tc = next((c for c in df_geomag.columns if 'time' in c.lower()), df_geomag.columns[0])
    kc = 'kp' if 'kp' in df_geomag.columns else next(
        (c for c in df_geomag.columns if 'kp' in c.lower() and c != tc), df_geomag.columns[1])
    df_geomag = df_geomag[[tc, kc]].copy()
    df_geomag.columns = ['time', 'kp']
    df_geomag['time'] = pd.to_datetime(df_geomag['time'])
    df_geomag['kp'] = df_geomag['kp'].astype(str).str.extract(r'([0-9]+(?:\.[0-9]*)?)')[0]
    df_geomag['kp'] = pd.to_numeric(df_geomag['kp'], errors='coerce')
    df_geomag = df_geomag[df_geomag['kp'] >= 0].dropna().sort_values('time').tail(1440)
    if not df_geomag.empty:
        print(f'Geomag Kp:  aktuell {df_geomag["kp"].iloc[-1]:.1f}')

print('\n✅ Aufbereitung abgeschlossen')

---
## 4. Visualisierungen

In [ ]:
# ============================================================
# SEISMIK – Magnitude-Zeitreihe + Tiefenverteilung
# ============================================================

if df_seis is not None:
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Erdbeben M≥4.5 (letzte 7 Tage)', 'Tiefenverteilung [km]'],
        column_widths=[0.65, 0.35]
    )

    # Farbskala nach Magnitude
    mag_colors = ['#2ecc71' if m < 5 else '#f39c12' if m < 6 else '#e74c3c'
                  for m in df_seis['mag']]
    sizes = [max(6, (m - 4) * 12) for m in df_seis['mag']]

    fig.add_trace(go.Scatter(
        x=df_seis['time'], y=df_seis['mag'],
        mode='markers',
        marker=dict(color=mag_colors, size=sizes, opacity=0.75,
                    line=dict(color='white', width=0.5)),
        text=df_seis['place'],
        hovertemplate='%{text}<br>M%{y:.1f}<br>%{x}<extra></extra>',
        name='Erdbeben'
    ), row=1, col=1)

    fig.add_hline(y=6.0, line_dash='dot', line_color='#e74c3c',
                  annotation_text='M6.0', row=1, col=1)
    fig.add_hline(y=5.0, line_dash='dot', line_color='#f39c12',
                  annotation_text='M5.0', row=1, col=1)

    # Tiefenverteilung
    depth_bins = pd.cut(df_seis['depth'],
                        bins=[0, 35, 70, 150, 300, 700],
                        labels=['0–35 km\n(Kruste)', '35–70 km',
                                '70–150 km', '150–300 km', '300–700 km\n(tief)'])
    depth_counts = depth_bins.value_counts().sort_index()
    depth_colors = ['#B4B2A9', '#888780', '#5F5E5A', '#3d3d3a', '#2C2C2A']

    fig.add_trace(go.Bar(
        x=depth_counts.values,
        y=depth_counts.index.astype(str),
        orientation='h',
        marker_color=depth_colors[:len(depth_counts)],
        opacity=0.85,
        name='Tiefe'
    ), row=1, col=2)

    fig.update_layout(
        title=dict(text='Seismische Aktivität – Layer 1 (USGS)', font=dict(size=15)),
        height=380, showlegend=False,
        plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=60, r=30, t=60, b=50)
    )
    fig.update_yaxes(title_text='Magnitude', row=1, col=1)
    fig.show()
else:
    print('Seismik-Daten nicht verfügbar.')

In [ ]:
# ============================================================
# SEISMIK – Weltkarte
# ============================================================

if df_seis is not None:
    mag_colors_map = ['#2ecc71' if m < 5 else '#f39c12' if m < 6 else '#e74c3c'
                      for m in df_seis['mag']]
    sizes_map = [max(4, (m - 4) * 10) for m in df_seis['mag']]

    fig = go.Figure(go.Scattergeo(
        lon=df_seis['lon'], lat=df_seis['lat'],
        mode='markers',
        marker=dict(color=mag_colors_map, size=sizes_map, opacity=0.7,
                    line=dict(color='white', width=0.3)),
        text=df_seis['place'],
        hovertemplate='%{text}<br>M' + df_seis['mag'].round(1).astype(str) + '<extra></extra>',
        name='M≥4.5'
    ))

    fig.update_geos(
        projection_type='natural earth',
        showland=True, landcolor='#2C2C2A',
        showocean=True, oceancolor='#0C447C',
        showcoastlines=True, coastlinecolor='#444441',
        showframe=False
    )
    fig.update_layout(
        title=dict(text='Seismische Aktivität M≥4.5 – letzte 7 Tage (USGS)', font=dict(size=14)),
        height=400,
        margin=dict(l=0, r=0, t=50, b=0),
        paper_bgcolor='rgba(0,0,0,0)'
    )

    # Legende
    for mag, color, label in [(4.5,'#2ecc71','M4.5–5.0'), (5.0,'#f39c12','M5.0–6.0'), (6.0,'#e74c3c','M≥6.0')]:
        fig.add_trace(go.Scattergeo(
            lon=[None], lat=[None], mode='markers',
            marker=dict(color=color, size=8),
            name=label, showlegend=True
        ))

    fig.show()

In [ ]:
# ============================================================
# ERDROTATION – LOD-Variation (falls IERS verfügbar)
# ============================================================

lod_col = None
if df_iers is not None:
    lod_col = next((c for c in df_iers.columns if 'lod' in c), None)

if df_iers is not None and lod_col:
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        y=df_iers[lod_col].values,
        mode='lines',
        line=dict(color='#888780', width=1.8),
        fill='tozeroy', fillcolor='rgba(136,135,128,0.15)',
        name='LOD'
    ))
    fig.add_hline(y=0, line_color='#5F5E5A', line_width=0.8)
    fig.update_layout(
        title=dict(text='Erdrotation – LOD-Variation (ms), letzte 90 Tage (IERS)', font=dict(size=14)),
        xaxis_title='Tage (rückwärts)', yaxis_title='ΔT Tageslänge [ms]',
        height=300,
        plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=60, r=30, t=50, b=40)
    )
    fig.show()
else:
    print('IERS LOD-Daten nicht verfügbar oder Spalte nicht gefunden.')
    print('Referenzwert: Nominale Tageslänge =', CONSTANTS['Siderischer_Tag_s'], 's')

In [ ]:
# ============================================================
# LEITFÄHIGKEITS-PROFIL – Erdoberfläche
# Statisches Balkendiagramm der Referenzwerte
# ============================================================

leitf_data = {
    'Meerwasser (Ozean)':       3.2,
    'Flusswasser':              0.05,
    'Feuchter Boden':           0.05,
    'Trockener Boden':          0.001,
    'Sedimentgestein':          0.01,
    'Kristallines Gestein':     1e-4,
    'Trockener Granit':         1e-6,
}

fig = go.Figure(go.Bar(
    x=list(leitf_data.values()),
    y=list(leitf_data.keys()),
    orientation='h',
    marker=dict(
        color=['#185FA5','#378ADD','#639922','#B4B2A9','#888780','#5F5E5A','#3d3d3a'],
        opacity=0.85
    )
))
fig.update_layout(
    title=dict(text='Elektrische Leitfähigkeit σ [S/m] – Layer-1-Materialien', font=dict(size=14)),
    xaxis=dict(type='log', title='σ [S/m] (logarithmisch)'),
    height=320,
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=180, r=30, t=50, b=40),
    showlegend=False
)
fig.show()

---
## 5. Zustandsbewertung & Übergabe an Layer 2

In [ ]:
# ============================================================
# LAYER-1-SCORE
# Vier Komponenten: Seismik, Rotation, Magnetfeld, Leitfähigkeit
# ============================================================

def norm(v, lo, hi):
    if v is None: return None
    return max(0.0, min(1.0, (v - lo) / (hi - lo)))

# --- Seismische Aktivität ---
seis_score_val = None
seis_source = 'missing'
if df_seis is not None and not df_seis.empty:
    n_events = len(df_seis)
    max_mag  = df_seis['mag'].max()
    n_strong = (df_seis['mag'] >= 6.0).sum()
    # Score: kombiniert Häufigkeit + Stärke
    seis_score_val = norm(n_events * 0.5 + max_mag * 5 + n_strong * 10, 0, 120)
    seis_source = 'primary'

# --- Erdrotation (LOD-Anomalie) ---
lod_score_val = None
lod_now       = None
lod_source    = 'missing'
if df_iers is not None and lod_col:
    lod_now = float(df_iers[lod_col].iloc[-1])
    lod_score_val = norm(abs(lod_now), 0, 3.0)
    lod_source = 'primary'
elif raw.get('iers_synthetic_lod') is not None:
    lod_now = raw['iers_synthetic_lod']
    lod_score_val = norm(abs(lod_now), 0, 3.0)
    lod_source = 'estimated_seasonal'
    print(f'  LOD Schätzwert: {lod_now:+.3f} ms  [estimated_seasonal]')

# --- Erdmagnetfeld-Variation (Kp als dynamischer Proxy) ---
mag_score_val = None
kp_val        = None
mag_source    = 'missing'
if df_geomag is not None and not df_geomag.empty:
    kp_val = float(df_geomag['kp'].iloc[-1])
    mag_score_val = norm(kp_val, 0, 9)
    mag_source = 'primary'
elif layer0 and layer0.get('raw_values', {}).get('Kp_index', {}).get('value') is not None:
    kp_val = layer0['raw_values']['Kp_index']['value']
    mag_score_val = norm(kp_val, 0, 9)
    mag_source = 'from_layer0'

# --- Leitfähigkeits-Baseline (statisch – nicht im dynamischen Score) ---
ocean_fraction = 0.71
leitf_baseline = {
    'ocean_surface_fraction': ocean_fraction,
    'ocean_conductivity_Sm':  CONSTANTS['Leitf_Ozeane_Sm'],
    'role': 'stable_boundary_condition',
}

# --- Zusammenfassen (nur dynamische Komponenten im Score) ---
COMPONENTS = {
    'Seismische Aktivität':      {'score': seis_score_val,  'source': seis_source,  'dynamic': True},
    'Erdrotation (LOD-Anomalie)':{'score': lod_score_val,   'source': lod_source,   'dynamic': True},
    'Magnetfeld-Variation (Kp)': {'score': mag_score_val,   'source': mag_source,   'dynamic': True},
}

available   = {k: v['score'] for k, v in COMPONENTS.items() if v['score'] is not None}
unavailable = [k for k, v in COMPONENTS.items() if v['score'] is None]

layer1_score = round(sum(available.values()) / len(available), 4) if available else None
confidence   = round(len(available) / len(COMPONENTS), 2)
level = ('unbekannt' if layer1_score is None
         else 'ruhig'   if layer1_score < 0.3
         else 'moderat' if layer1_score < 0.6
         else 'aktiv')

# --- Ausgabe Konsole ---
W = 64
print('=' * W)
print('LAYER 1 – GEOPHYSIKALISCHER GRUNDZUSTAND – ZUSTANDSBEWERTUNG')
print('=' * W)
for name, comp in COMPONENTS.items():
    s = comp['score']
    if s is not None:
        bar = '█' * int(s * 20) + '░' * (20 - int(s * 20))
        print(f'  {name:<34} {bar}  {s:.2f}  [{comp["source"]}]')
    else:
        print(f'  {name:<34} {"─" * 20}  n/a   [missing]')
print(f'  {"Leitfähigkeit (Baseline)":<34} {"─" * 20}  static [boundary_condition]')
print('-' * W)
print(f'  Score (dynamisch):  {layer1_score:.3f}  ({len(available)}/{len(COMPONENTS)} Komponenten)')
print(f'  Confidence:         {confidence:.0%}')
print(f'  Level:              {level.upper()}')
if layer0:
    print(f'  Layer-0-Modulator:  {layer0["level"].upper()} | Driver: {layer0["dominant_driver"]}')
print('=' * W)

# Radar (nur dynamische)
cats = list(available.keys())
vals = list(available.values())
if len(cats) >= 3:
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals + [vals[0]], theta=cats + [cats[0]],
        fill='toself', fillcolor='rgba(136,135,128,0.22)',
        line=dict(color='#888780', width=2.5), name='Layer 1 (dynamisch)'
    ))
    fig.add_trace(go.Scatterpolar(
        r=[0.5] * (len(cats)+1), theta=cats + [cats[0]],
        line=dict(color='#E24B4A', dash='dot', width=1),
        mode='lines', name='Aktivitätsschwelle'
    ))
    fig.update_layout(
        title=dict(
            text=f'Layer 1 – Dynamisches Aktivitätsprofil | Score: {layer1_score:.2f} | {level.upper()} | Confidence: {confidence:.0%}',
            font=dict(size=13)),
        polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
        height=430, showlegend=True,
        margin=dict(l=60, r=60, t=65, b=40)
    )
    fig.show()

In [ ]:
# ============================================================
# EXPORT – layer1_state.json
# ============================================================

# state_summary
_seis_str = (
    f'Seismik: {len(df_seis)} Ereignisse M>=4.5 in 7 Tagen, staerkste M{df_seis["mag"].max():.1f}.'
    if df_seis is not None and not df_seis.empty
    else 'Seismik-Daten nicht verfuegbar.'
)
_lod_src  = COMPONENTS['Erdrotation (LOD-Anomalie)']['source']
_lod_str  = (f'LOD-Anomalie: {lod_now:+.3f} ms [{_lod_src}].'
             if lod_now is not None else 'Erdrotation (LOD): nicht verfuegbar.')
_mag_str  = (f'Magnetfeld-Variation Kp={kp_val:.1f}.'
             if kp_val is not None else 'Magnetfeld-Variation: nicht verfuegbar.')
_l0_str   = (f'Layer-0-Modulator: {layer0["level"]} (Score {layer0["score"]:.2f}).'
             if layer0 else 'Layer-0-Kontext fehlt.')

state_summary = ' '.join([
    f'Layer-1-Zustand: {level}.', _seis_str, _lod_str, _mag_str,
    f'Datenvollstaendigkeit: {confidence:.0%}.', _l0_str
])

# Schumann-Formulierung abhängig von Kp
_schumann = (
    'Grundmode stabil ~7.83 Hz – keine Modulation aus Layer 1 erwartet'
    if kp_val is None or kp_val < 3
    else 'nur geringe indirekte Modulation moeglich (Kp < 5, kein starker Layer-1-Treiber)'
    if kp_val < 5
    else 'erhoehte Magnetfeldvariation – schwache Modulation moeglich'
)

layer1_state = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'layer': 1,
    'name':  'Geophysikalischer Grundzustand',

    # Score & Qualitaet (nur dynamische Komponenten)
    'score':      layer1_score,
    'level':      level,
    'confidence': confidence,
    'score_basis': f'{len(available)}/{len(COMPONENTS)} dynamische Komponenten verfuegbar',
    'dynamic_score_note': 'Score basiert ausschliesslich auf dynamischen Komponenten. Leitfaehigkeits-Baseline ist stabile Randbedingung, nicht im Score.',
    'missing_components': unavailable,

    # Dynamische Komponenten mit dynamic-Flag
    'components': {
        k: {
            'score':   round(v['score'], 4) if v['score'] is not None else None,
            'source':  v['source'],
            'dynamic': v['dynamic'],
        }
        for k, v in COMPONENTS.items()
    },

    # Statische Baseline (nicht im Score)
    'baseline_properties': leitf_baseline,

    # Physikalische Konstanten (Referenz)
    'reference_constants': {
        'Erdmasse_kg':         CONSTANTS['Erdmasse_kg'],
        'Rotationsperiode_h':  CONSTANTS['Rotationsperiode_h'],
        'Dipolmoment_Am2':     CONSTANTS['Dipolmoment_Am2'],
        'Schumann_Ref_Hz':     CONSTANTS['Schumann_Grundmode_Hz'],
    },

    # Rohdaten
    'raw_values': {
        'seismic_events_7d':   len(df_seis) if df_seis is not None else None,
        'seismic_max_mag':     round(float(df_seis['mag'].max()), 1) if df_seis is not None else None,
        'seismic_M6plus':      int((df_seis['mag'] >= 6.0).sum()) if df_seis is not None else None,
        'LOD_anomaly_ms':      round(lod_now, 4) if lod_now is not None else None,
        'LOD_source':          COMPONENTS['Erdrotation (LOD-Anomalie)']['source'],
        'Kp_index':            round(kp_val, 1) if kp_val is not None else None,
    },

    # Flags
    'flags': {
        'strong_earthquake':   (float(df_seis['mag'].max()) >= 6.5) if df_seis is not None else None,
        'elevated_seismicity': (len(df_seis) > 50)                  if df_seis is not None else None,
        'mag_field_disturbed': (kp_val >= 4)                        if kp_val  is not None else None,
        'lod_anomaly_high':    (abs(lod_now) > 1.5)                 if lod_now is not None else None,
    },

    # Downstream-Erwartung
    'downstream_expectation': {
        'layer2_surface':    ('erhoehte Spannung im Untergrund moeglich'
                              if df_seis is not None and float(df_seis['mag'].max()) >= 6.0
                              else 'normal'),
        'layer4_ionosphere': ('Ionosphaeren-Modifikation durch Kp moeglich'
                              if kp_val is not None and kp_val >= 4
                              else 'normal'),
        'layer5_gec':        'Grundleitfaehigkeit stabil – 71% Ozeane (Randbedingung)',
        'schumann_resonance': _schumann,
    },

    # Layer-0-Kontext
    'layer0_context': {
        'score':           layer0['score']           if layer0 else None,
        'level':           layer0['level']           if layer0 else None,
        'dominant_driver': layer0['dominant_driver'] if layer0 else None,
        'confidence':      layer0['confidence']      if layer0 else None,
    },

    'state_summary': state_summary,
}

with open(layer_state(1), 'w', encoding='utf-8') as f:
    json.dump(layer1_state, f, indent=2, ensure_ascii=False)

print(f'✅ gespeichert: {layer_state(1)}')
print(json.dumps(layer1_state, indent=2, ensure_ascii=False))

---
## Zusammenfassung Layer 1

| Aspekt | Inhalt |
|--------|--------|
| **Rolle** | Materieller Träger – Resonanzraum, Leitfähigkeitsstruktur, Feldanker |
| **Konstanten** | Erdmasse, Rotation, IGRF-Dipolmoment, Schumann-Grundmode |
| **Dynamische Daten** | USGS Seismik, IERS LOD, NOAA Kp |
| **→ Layer 2** | Seismische Spannungen, Bodenfeuchte-Leitfähigkeit |
| **→ Layer 4** | Magnetfeld-Kp → Ionosphärische Modifikation |
| **→ Layer 5** | Ozean-Leitfähigkeit (71%) → Grundstruktur des GEC |
| **→ Layer 6** | Schumann-Grundmode 7.83 Hz als Referenz |
| **Ausgabe** | `layer1_state.json` mit Layer-0-Kontext |

> **Nächster Schritt:** `atmosphere_analysis_layer2.ipynb` – Erdoberfläche / Ozeane / Land